# Runner — Anatomy-Aware Pose Estimation (STL Training)
Notebook **minimale**: tutto il codice vero sta nei moduli `.py` sul repo GitHub.

**Prima di lanciare:**
1. Settings → **Internet: On**
2. **Add Input** → COCO 2017 Keypoints + OCHuman + dataset `pose-baseline-checkpoint`

In [6]:
# === 1. Codice dal repo GitHub ===
import os, sys

REPO_URL = "https://github.com/flaviomassaroni/Anatomy-Aware-Pose-Estimation-on-Lightweight-Networks.git"
REPO_DIR = "/kaggle/working/repo"

!rm -rf {REPO_DIR}
!git clone -q {REPO_URL} {REPO_DIR}

mod_dir = None
for root, dirs, files in os.walk(REPO_DIR):
    if ".git" in root: continue
    if "config.py" in files:
        mod_dir = root
        break
if mod_dir:
    sys.path.insert(0, mod_dir)
    print("Moduli:", sorted([f for f in os.listdir(mod_dir) if f.endswith(".py")]))
else:
    print("ERRORE: config.py non trovato!")

fatal: could not create leading directories of '/kaggle/working/repo': Permission denied
ERRORE: config.py non trovato!


In [7]:
import stl, inspect
print("BONE_SCALE" in dir(stl))                    # True solo col file nuovo
print("_logcosh" in dir(stl))                       # idem
src = inspect.getsource(stl.bone_ratio_loss)
print("logcosh" in src and "log-cosh" in src)       # True col nuovo

True
True
True


In [11]:
# === 2. Import + seed + check ===
import utils, data, network, train
import evaluation as ev
from stl import SkeletalTopologyLoss
import cv2
cv2.setNumThreads(0)  # Forza OpenCV a non spawnare thread propri

import config, importlib
importlib.reload(config)
print(config.__file__)
print(hasattr(config, "ENV_NAME"))

set_seed(SEED)
print("Device:", device)
print("Ambiente:", ENV_NAME)
print("COCO val ann:", COCO_VAL_ANN)

# Listing dataset montati: solo su Kaggle. In locale /kaggle/input non esiste.
if os.path.isdir("/kaggle/input"):
    print("\n/kaggle/input contiene:")
    for d in os.listdir("/kaggle/input"):
        print("  -", d)
else:
    print("\n(locale: /kaggle/input non presente, uso path da config)")

/home/flavio/Documents/University/Computer vision/Anatomy-Aware-Pose-Estimation-on-Lightweight-Networks/anatomy-aware-pose/config.py
True
Device: cuda


NameError: name 'ENV_NAME' is not defined

In [ ]:
# === 3. Dati COCO ===
from torch.utils.data import DataLoader

train_samples = data.build_samples(COCO_TRAIN_ANN, min_keypoints=8)
val_samples   = data.build_samples(COCO_VAL_ANN)
print(f"train: {len(train_samples)} | val: {len(val_samples)}")

train_dataset = data.COCOKeypointsDataset(train_samples, COCO_TRAIN_IMG, INPUT_SIZE, HEATMAP_SIZE, SIGMA, NUM_KEYPOINTS)
val_dataset   = data.COCOKeypointsDataset(val_samples,   COCO_VAL_IMG,   INPUT_SIZE, HEATMAP_SIZE, SIGMA, NUM_KEYPOINTS)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=32, shuffle=False, num_workers=2, pin_memory=True)

In [ ]:
BEST_PTH = "/kaggle/input/datasets/messinaalberto/pose-baseline-checkpoint/best.pth"
print("Esiste:", os.path.exists(BEST_PTH))

In [ ]:
if False:
    import torch
    from stl import soft_argmax
    model.eval()
    ratios_all = []
    valids_all = []
    with torch.no_grad():
        for bi,(imgs,hms,w) in enumerate(train_loader):
            if bi>=8: break
            imgs,w = imgs.to(device), w.to(device)
            c = soft_argmax(model(imgs), beta=10.0)
            def blen(c,a,b): return torch.sqrt(((c[:,a]-c[:,b])**2).sum(-1)+1e-6)
            r = blen(c,7,9)/(blen(c,5,7)+1e-6)
            v = (w.squeeze(-1)[:,5]>0)&(w.squeeze(-1)[:,7]>0)&(w.squeeze(-1)[:,9]>0)
            ratios_all.append(r[v].cpu())
    r = torch.cat(ratios_all)
    qs = torch.quantile(r, torch.tensor([0.01,0.05,0.25,0.5,0.75,0.95,0.99]))
    print("quantili forearm/upper_arm (solo validi):")
    for p,q in zip([1,5,25,50,75,95,99], qs): print(f"  p{p:02d}: {q:.3f}")

In [ ]:
# === 3c. Calibrazione lambda gradient-norm (Strada B) ===
# Misura ||grad L_t / d(final_layer)|| per ogni termine NON pesato
# e per la heatmap loss. Lambda calibrati come:
#   lambda_t = STL_TARGET_FRAC * g_hm / (g_t + eps)
# Cosi ogni termine imprime ai pesi una frazione TARGET_FRAC della
# spinta della heatmap loss (influenza equalizzata, non valore equalizzato).
# Ref: Chen et al. GradNorm 2018. Vedi stl.py::calibrate_lambdas per i dettagli.
import torch
from stl import SkeletalTopologyLoss, calibrate_lambdas

model = network.PoseMobileNet(NUM_KEYPOINTS, INPUT_SIZE, pretrained=False).to(device)
model.load_state_dict(torch.load(BEST_PTH, map_location=device))

diag_criterion = SkeletalTopologyLoss(
    heatmap_criterion=train.WeightedMSELoss(),
    lambda_bone=1.0, lambda_angle=1.0, lambda_order=1.0, lambda_collapse=1.0,
    beta=STL_BETA,
)

# Se COCO train non e'ancora scaricato, campiona dal val_loader (1 riga)
try:
    calib_loader = train_loader
except NameError:
    calib_loader = val_loader
    print("ATTENZIONE: uso val_loader per calibrazione (COCO train non disponibile)")

lam_suggest = calibrate_lambdas(
    diag_criterion, model, calib_loader, device,
    target_frac=STL_TARGET_FRAC, n_batches=4, verbose=True,
)
# calibrate_lambdas stampa norme grad + lambda calibrati.
# lam_suggest dict viene letto dalla cella 4b.


In [ ]:
# === 4a. Training BASELINE (gia' completato — NON eseguire) ===
if False:
    import torch
    NUM_EPOCHS = 30
    model = network.PoseMobileNet(NUM_KEYPOINTS, INPUT_SIZE, pretrained=True).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    scheduler = torch.optim.lr_scheduler.MultiStepLR(optimizer, milestones=[15, 25], gamma=0.1)
    criterion = train.WeightedMSELoss()
    history = train.fit(model, train_loader, val_loader, optimizer, scheduler, criterion, device, NUM_EPOCHS, CHECKPOINT_DIR, resume=True)

In [ ]:
# === 4b-SANITY. Sanity check 1 epoca PRIMA del run completo ===
# Esegue UNA epoca di fine-tuning STL da best.pth e verifica due guardie
# (sez. 12.1 del README) contro la baseline:
#   G1) AP deve reggere entro ~1 punto:  AP >= 0.488   (baseline 0.498)
#   G2) AVR deve gia' scendere:          AVR < 0.306   (baseline 0.306)
# Se una guardia fallisce, NON lanciare la cella 9: qualcosa nel harness
# (lambda, LR, o gate STL/AVR) sta ancora rompendo il training.
# Questa epoca parte da best.pth e salva in una dir separata: NON sporca
# il run definitivo, che ricomincia comunque da best.pth.
import torch, os, time
from tqdm.auto import tqdm
import evaluation as ev
from stl import SkeletalTopologyLoss, calibrate_lambdas

# --- Baseline di riferimento (dal README, sez. 10/11.9) ---
BASE_AP  = 0.498
BASE_AVR = 0.306
AP_TOL   = 0.010          # "entro ~1 punto"
AP_FLOOR = BASE_AP - AP_TOL   # 0.488

# --- BEST_PTH portabile (stessa logica della cella 9) ---
try:
    _best = BEST_PTH
except NameError:
    _best = None
if not _best or not os.path.exists(_best):
    _best = "/kaggle/input/datasets/messinaalberto/pose-baseline-checkpoint/best.pth"
assert os.path.exists(_best), f"best.pth non trovato: {_best}"
print("Baseline checkpoint:", _best)

model = network.PoseMobileNet(NUM_KEYPOINTS, INPUT_SIZE, pretrained=False).to(device)
model.load_state_dict(torch.load(_best, map_location=device))
print("Pesi baseline caricati")

LR_STL = STL_FINE_TUNE_LR

# --- Calibrazione lambda (identica alla cella 9, sul modello baseline) ---
criterion = SkeletalTopologyLoss(
    heatmap_criterion=train.WeightedMSELoss(),
    lambda_bone=1.0, lambda_angle=1.0, lambda_order=1.0, lambda_collapse=1.0,
    beta=STL_BETA,
)
print("Calibrazione lambda (gradient-norm, su baseline)...")
lam = calibrate_lambdas(
    criterion, model, train_loader, device,
    target_frac=STL_TARGET_FRAC, n_batches=4, verbose=True,
)
print(f"  lambda: bone={criterion.lambda_bone:.5f} angle={criterion.lambda_angle:.5f} "
      f"order={criterion.lambda_order:.5f} collapse={criterion.lambda_collapse:.5f}")

# Azzero i gradienti sporchi lasciati dalla calibrazione PRIMA dell'optimizer
optimizer = torch.optim.Adam(model.parameters(), lr=LR_STL)
optimizer.zero_grad(set_to_none=True)

SANITY_DIR = "/kaggle/working/checkpoints_sanity"
os.makedirs(SANITY_DIR, exist_ok=True)

# --- UNA epoca di training ---
t0 = time.time()
model.train()
sums = {k: 0.0 for k in ['heatmap', 'bone', 'angle', 'order', 'collapse', 'total']}
for imgs, hms, w in tqdm(train_loader, desc="sanity 1 epoca", leave=False):
    imgs, hms, w = imgs.to(device), hms.to(device), w.to(device)
    optimizer.zero_grad()
    out = model(imgs)
    loss, terms = criterion(out, hms, w)
    loss.backward()
    optimizer.step()
    for k in sums: sums[k] += terms[k] * imgs.size(0)
n = len(train_loader.dataset)
for k in sums: sums[k] /= n

torch.save(model.state_dict(), f"{SANITY_DIR}/sanity_e01.pth")

# --- Valutazione AP + AVR su COCO val ---
model.eval()
coco_eval, avr = ev.evaluate_on_coco_val(
    model, val_samples, device,
    results_path=f"{SANITY_DIR}/coco_val_pred_sanity.json")
ap  = float(coco_eval.stats[0])
ar  = float(coco_eval.stats[5])
avr_rate = avr['AVR_pose_rate']

print(f"\nE01 (sanity) tot:{sums['total']:.4f} hm:{sums['heatmap']:.4f} "
      f"bone:{sums['bone']:.4f} ang:{sums['angle']:.4f} "
      f"ord:{sums['order']:.4f} col:{sums['collapse']:.4f} "
      f"| AP:{ap:.4f} AR:{ar:.4f} AVR:{avr_rate:.4f} | {time.time()-t0:.0f}s")

# --- VERDETTO ---
g1 = ap >= AP_FLOOR
g2 = avr_rate < BASE_AVR
print("\n" + "=" * 56)
print("VERDETTO SANITY CHECK")
print("=" * 56)
print(f"  G1  AP regge      : AP {ap:.4f} >= {AP_FLOOR:.4f}  -> "
      f"{'PASS' if g1 else 'FAIL'}   (baseline {BASE_AP:.4f})")
print(f"  G2  AVR scende    : AVR {avr_rate:.4f} < {BASE_AVR:.4f} -> "
      f"{'PASS' if g2 else 'FAIL'}   (baseline {BASE_AVR:.4f})")
print("-" * 56)
if g1 and g2:
    print("  ESITO: PASS — puoi lanciare la cella 9 (run completo).")
    print(f"         AP {BASE_AP:.4f}->{ap:.4f} ({ap-BASE_AP:+.4f}), "
          f"AVR {BASE_AVR:.4f}->{avr_rate:.4f} ({avr_rate-BASE_AVR:+.4f})")
else:
    print("  ESITO: FAIL — NON lanciare il run completo. Diagnosi:")
    if not g1:
        print("    - AP crollato in 1 epoca: LR/lambda troppo aggressivi, la STL")
        print("      sta distruggendo le heatmap (catastrophic forgetting).")
    if not g2:
        print("    - AVR non scende (o sale): sintomo del mismatch gate STL/AVR")
        print("      (STL su target_weight GT, AVR su score>=MIN_CONF predetto).")
        print("      La loss migliora le coord soft-argmax sui kp annotati mentre")
        print("      le coord MISURATE (argmax) peggiorano. Fermati e rivedi il gate.")
print("=" * 56)


In [ ]:
# === 4b. Fine-tuning con STL (calibrazione lambda integrata + selezione su AP COCO val) ===
import torch, os, time
from tqdm.auto import tqdm
import evaluation as ev
from stl import SkeletalTopologyLoss, calibrate_lambdas

# --- BEST_PTH portabile: rispetta quello del config se esiste, altrimenti
#     cade sul path Kaggle del dataset condiviso. Cosi la stessa cella gira
#     sia in locale (config locale definisce BEST_PTH) sia su Kaggle. ---
try:
    _best = BEST_PTH
except NameError:
    _best = None
if not _best or not os.path.exists(_best):
    _best = "/kaggle/input/datasets/messinaalberto/pose-baseline-checkpoint/best.pth"
assert os.path.exists(_best), f"best.pth non trovato: {_best}"
print("Baseline checkpoint:", _best)

model = network.PoseMobileNet(NUM_KEYPOINTS, INPUT_SIZE, pretrained=False).to(device)
model.load_state_dict(torch.load(_best, map_location=device))
print('Pesi baseline caricati')

# --- Iperparametri fine-tuning (tutti da config.py) ---
NUM_EPOCHS_STL = STL_NUM_EPOCHS
LR_STL = STL_FINE_TUNE_LR   # 3e-5: evita catastrophic forgetting (era 1e-4)

# --- Calibrazione lambda INTEGRATA (non dipende piu' dalla cella 3c) ---
# Misura ||grad L_t / d(final_layer)|| per ogni termine non pesato e per la
# heatmap loss, poi lambda_t = STL_TARGET_FRAC * g_hm / (g_t + eps), con spread
# clamp (max 20x) per stabilita'. Vedi stl.py::calibrate_lambdas.
# La eseguo SUL MODELLO BASELINE appena caricato, prima del fine-tuning, cosi
# i lambda riflettono lo stato da cui partiamo. Se lam_suggest esiste gia'
# (cella 3c eseguita a mano), lo rispetto e salto la ricalibrazione.
criterion = SkeletalTopologyLoss(
    heatmap_criterion=train.WeightedMSELoss(),
    lambda_bone=1.0, lambda_angle=1.0, lambda_order=1.0, lambda_collapse=1.0,
    beta=STL_BETA,
)

try:
    lam = lam_suggest  # gia' calibrato dalla cella 3c
    criterion.lambda_bone     = lam['bone']
    criterion.lambda_angle    = lam['angle']
    criterion.lambda_order    = lam['order']
    criterion.lambda_collapse = lam['collapse']
    print("Uso lambda gia' calibrati dalla cella 3c")
except NameError:
    print('Calibrazione lambda in corso (gradient-norm, su baseline)...')
    lam = calibrate_lambdas(
        criterion, model, train_loader, device,
        target_frac=STL_TARGET_FRAC, n_batches=4, verbose=True,
    )
    # calibrate_lambdas aggiorna criterion in-place e ritorna i lambda.

print(f"  lambda finali: bone={criterion.lambda_bone:.5f} "
      f"angle={criterion.lambda_angle:.5f} order={criterion.lambda_order:.5f} "
      f"collapse={criterion.lambda_collapse:.5f}")

# IMPORTANTE: calibrate_lambdas lascia gradienti sporchi sul modello.
# Azzero prima di creare l'optimizer e iniziare il training.
optimizer = torch.optim.Adam(model.parameters(), lr=LR_STL)
optimizer.zero_grad(set_to_none=True)

# milestone a 70% delle epoche: si scala automaticamente se cambi NUM_EPOCHS_STL
scheduler = torch.optim.lr_scheduler.MultiStepLR(
    optimizer, milestones=[round(0.7 * NUM_EPOCHS_STL)], gamma=0.1)

STL_CKPT_DIR = '/kaggle/working/checkpoints_stl'
os.makedirs(STL_CKPT_DIR, exist_ok=True)

# --- Criterio di selezione: AP COCO val, NON val_loss ---
# val_loss include la STL: minimizzarla premia un modello con STL bassa anche
# se l'AP e' peggiorata. L'obiettivo del progetto e' abbassare AVR SENZA
# distruggere AP, quindi seleziono sull'AP (stats[0] di pycocotools).
# Salvo comunque ogni epoca, cosi la scelta finale resta ispezionabile a mano.
best_ap = -1.0
history = []

for epoch in range(1, NUM_EPOCHS_STL + 1):
    t0 = time.time()
    model.train()
    sums = {k: 0.0 for k in ['heatmap', 'bone', 'angle', 'order', 'collapse', 'total']}
    for imgs, hms, w in tqdm(train_loader, desc=f'train {epoch}', leave=False):
        imgs, hms, w = imgs.to(device), hms.to(device), w.to(device)
        optimizer.zero_grad()
        out = model(imgs)
        loss, terms = criterion(out, hms, w)
        loss.backward()
        optimizer.step()
        for k in sums: sums[k] += terms[k] * imgs.size(0)
    n = len(train_loader.dataset)
    for k in sums: sums[k] /= n

    # Salvo SEMPRE il checkpoint di questa epoca (scelta finale ispezionabile)
    torch.save(model.state_dict(), f'{STL_CKPT_DIR}/epoch_{epoch:02d}.pth')

    # Valuto AP+AVR su COCO val per la selezione
    model.eval()
    coco_eval, avr = ev.evaluate_on_coco_val(
        model, val_samples, device,
        results_path=f'{STL_CKPT_DIR}/coco_val_pred_e{epoch:02d}.json')
    ap = float(coco_eval.stats[0])      # AP @ IoU=0.50:0.95
    ar = float(coco_eval.stats[5])      # AR @ IoU=0.50:0.95
    avr_rate = avr['AVR_pose_rate']
    scheduler.step()
    lr = optimizer.param_groups[0]['lr']

    print(f"E{epoch:02d} tot:{sums['total']:.4f} hm:{sums['heatmap']:.4f} "
          f"bone:{sums['bone']:.4f} ang:{sums['angle']:.4f} "
          f"ord:{sums['order']:.4f} col:{sums['collapse']:.4f} "
          f"| AP:{ap:.4f} AR:{ar:.4f} AVR:{avr_rate:.4f} "
          f"lr:{lr:.0e} {time.time()-t0:.0f}s")
    history.append({'epoch': epoch, 'AP': ap, 'AR': ar, 'AVR': avr_rate,
                    **{f'L_{k}': sums[k] for k in sums}})

    if ap > best_ap:
        best_ap = ap
        torch.save(model.state_dict(), f'{STL_CKPT_DIR}/best_stl.pth')
        print(f'  -> nuovo best STL (AP {ap:.4f}, AVR {avr_rate:.4f})')

print('\nFine-tuning completato.')
print('Storia per-epoca (scegli a mano il trade-off AP/AVR che preferisci):')
for h in history:
    print(f"  E{h['epoch']:02d}  AP {h['AP']:.4f}  AR {h['AR']:.4f}  AVR {h['AVR']:.4f}")

# NB: best_stl.pth = miglior AP. Se un'epoca con AP di poco inferiore ha un AVR
# molto migliore, e' un trade-off legittimo per il paper: i checkpoint epoch_NN.pth
# sono tutti su disco per ricaricare quello che preferisci.


In [ ]:
# === 5. Valutazione: confronto baseline vs STL ===
import torch

model_base = network.PoseMobileNet(NUM_KEYPOINTS, INPUT_SIZE, pretrained=False).to(device)
model_base.load_state_dict(torch.load(BEST_PTH, map_location=device))
print("=== BASELINE ===")
coco_base, avr_coco_base = ev.evaluate_on_coco_val(model_base, val_samples, device)
oc_base, avr_oc_base = ev.evaluate_on_ochuman(model_base, device)

model_stl = network.PoseMobileNet(NUM_KEYPOINTS, INPUT_SIZE, pretrained=False).to(device)
model_stl.load_state_dict(torch.load(f'{STL_CKPT_DIR}/best_stl.pth', map_location=device))
print("\n=== STL ===")
coco_stl, avr_coco_stl = ev.evaluate_on_coco_val(model_stl, val_samples, device,
    results_path='/kaggle/working/coco_val_pred_stl.json')
oc_stl, avr_oc_stl = ev.evaluate_on_ochuman(model_stl, device,
    results_path='/kaggle/working/ochuman_pred_stl.json')

LABELS = ["AP","AP.50","AP.75","AP_M","AP_L","AR","AR.50","AR.75","AR_M","AR_L"]
header = f"{'':15}{'COCO base':>10}{'COCO STL':>10}{'':3}{'OC base':>10}{'OC STL':>10}"
print(f"\n{header}")
print("-" * 58)
for i, lab in enumerate(LABELS):
    cb = coco_base.stats[i]; cs = coco_stl.stats[i]
    ob = oc_base.stats[i]; os_ = oc_stl.stats[i]
    print(f"{lab:15}{cb:10.4f}{cs:10.4f}{'':3}{ob:10.4f}{os_:10.4f}")
ab = avr_coco_base['AVR_pose_rate']; as_ = avr_coco_stl['AVR_pose_rate']
ob = avr_oc_base['AVR_pose_rate']; os_ = avr_oc_stl['AVR_pose_rate']
print(f"\n{'AVR rate':15}{ab:10.4f}{as_:10.4f}{'':3}{ob:10.4f}{os_:10.4f}")
ab = avr_coco_base['AVR_mean_violations']; as_ = avr_coco_stl['AVR_mean_violations']
ob = avr_oc_base['AVR_mean_violations']; os_ = avr_oc_stl['AVR_mean_violations']
print(f"{'AVR mean':15}{ab:10.4f}{as_:10.4f}{'':3}{ob:10.4f}{os_:10.4f}")